# Phase 5 — Bag-of-Words & TF-IDF Vectorization

**Project:** Food Review Analyzer

This notebook converts cleaned review text into numerical features for machine learning.

Workflow:
`yelp_reviews_nlp.csv → train/test split → Bag-of-Words → TF-IDF`

Important:
- Do **not** use `stars_review` as a model feature because it directly determines the sentiment label.
- Use `clean_text` as the NLP input and `rating_review` as the target.


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("Phase 5 imports loaded.")


Phase 5 imports loaded.


## 1. Load the NLP dataset

In [3]:
file_path = "../data/processed/yelp_reviews_nlp.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print(df[["clean_text", "rating_review"]].head())
print("\nMissing clean_text:", df["clean_text"].isna().sum())
print("Missing rating_review:", df["rating_review"].isna().sum())


Dataset shape: (14351, 12)
                                          clean_text rating_review
0  decide eat aware going take hour beginning end...       Neutral
1  family diner buffet eclectic assortment large ...       Neutral
2  wow yummy different delicious favorite lamb cu...      Positive
3  cute interior owner gave u tour upcoming patio...      Positive
4  long term frequent customer establishment went...      Negative

Missing clean_text: 0
Missing rating_review: 0


## 2. Define input and target

In [4]:
X = df["clean_text"].fillna("")
y = df["rating_review"]

print("Input samples:", len(X))
print("Target classes:")
print(y.value_counts())


Input samples: 14351
Target classes:
rating_review
Positive    9857
Negative    2699
Neutral     1795
Name: count, dtype: int64


## 3. Train/test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


Training samples: 11480
Testing samples : 2871


## 4. Bag-of-Words

Bag-of-Words represents each review using word/phrase counts.

We use `ngram_range=(1, 2)` so the model can learn both:
- unigrams: `good`
- bigrams: `very good`


In [6]:
bow_vectorizer = CountVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print("BoW training shape:", X_train_bow.shape)
print("BoW testing shape :", X_test_bow.shape)


BoW training shape: (11480, 20000)
BoW testing shape : (2871, 20000)


## 5. Inspect Bag-of-Words vocabulary

In [7]:
bow_features = bow_vectorizer.get_feature_names_out()

print("Number of BoW features:", len(bow_features))
print("First 30 features:")
print(bow_features[:30])


Number of BoW features: 20000
First 30 features:
['aaron' 'aback' 'abc' 'ability' 'abita' 'able' 'able accommodate'
 'able add' 'able eat' 'able enjoy' 'able find' 'able finish' 'able get'
 'able give' 'able grab' 'able make' 'able order' 'able sit' 'able take'
 'able try' 'able walk' 'absence' 'absent' 'absolute' 'absolute best'
 'absolute favorite' 'absolute worst' 'absolutely' 'absolutely amazing'
 'absolutely awesome']


## 6. TF-IDF

TF-IDF gives higher importance to words that are useful for a document but less common across the entire dataset.


In [8]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape :", X_test_tfidf.shape)


TF-IDF training shape: (11480, 20000)
TF-IDF testing shape : (2871, 20000)


## 7. Inspect TF-IDF features

In [9]:
tfidf_features = tfidf_vectorizer.get_feature_names_out()

mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top_indices = mean_tfidf.argsort()[-20:][::-1]

top_tfidf = pd.DataFrame({
    "feature": tfidf_features[top_indices],
    "mean_tfidf": mean_tfidf[top_indices]
})

print("Number of TF-IDF features:", len(tfidf_features))

top_tfidf


Number of TF-IDF features: 20000


,feature,mean_tfidf
0,not,0.029661
1,food,0.026427
2,good,0.023968
3,great,0.023479
4,place,0.023336
5,service,0.019010
6,time,0.016160
7,like,0.014276
8,get,0.013942
9,back,0.013900


## 8. Save the TF-IDF vectorizer

In [10]:
import joblib

model_dir = "../model"
os.makedirs(model_dir, exist_ok=True)

vectorizer_path = os.path.join(model_dir, "tfidf_vectorizer.joblib")
joblib.dump(tfidf_vectorizer, vectorizer_path)

print("Saved:", vectorizer_path)


Saved: ../model\tfidf_vectorizer.joblib


## Phase 5 conclusion

We converted review text into numerical features using:
- Bag-of-Words
- TF-IDF

The TF-IDF vectorizer is saved in `model/`.

**Next:** Phase 6 trains and evaluates sentiment classification models.
